# บทเรียน 10 - ตัวแทน AI ในการใช้งานจริง

ในบทเรียนนี้คุณจะได้เรียนรู้ **รูปแบบการนำไปใช้ในระบบการผลิต** สำหรับตัวแทน AI โดยใช้ Microsoft Agent Framework (Python). เราจะครอบคลุม:

- **การสังเกตการณ์** — เพิ่มการจับเวลาและการบันทึกการโต้ตอบของตัวแทน
- **การประเมินผล** — ใช้ตัวแทนผู้ประเมินเพื่อให้คะแนนคุณภาพของการตอบสนอง
- **การจัดการค่าใช้จ่าย** — กลยุทธ์เพื่อการเพิ่มประสิทธิภาพโทเค็นและการเลือกโมเดล

สถานการณ์คือ **ตัวแทนท่องเที่ยว** ที่ช่วยผู้ใช้วางแผนการเดินทาง โดยมีการติดตามและการประเมินผลซ้อนอยู่ด้านบน.


## การตั้งค่า


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## ข้อควรพิจารณาในการนำไปใช้งานจริง

การย้ายเอเยนต์ AI จากต้นแบบไปสู่การใช้งานจริงจำเป็นต้องให้ความสนใจกับสามเสาหลักอย่างรอบคอบ:

1. **Observability** — คุณต้องมีการมองเห็นว่าตัวเอเยนต์กำลังทำอะไร ใช้เวลานานเท่าใด และเรียกใช้เครื่องมือใดบ้าง หากไม่มีการติดตามและการบันทึก การดีบักปัญหาในสภาพแวดล้อมการผลิตแทบเป็นไปไม่ได้

2. **Evaluation** — การตรวจสอบคุณภาพอัตโนมัติช่วยให้มั่นใจว่าคำตอบของเอเยนต์ยังคงถูกต้อง ครบถ้วน และเป็นประโยชน์เมื่อเวลาผ่านไป ตัวเอเยนต์ผู้ประเมินสามารถให้คะแนนคำตอบตามเกณฑ์ที่กำหนดได้

3. **Cost Management** — การใช้งานโทเค็นส่งผลโดยตรงต่อค่าใช้จ่าย ยุทธศาสตร์เช่นการปรับแต่งพรอมต์ การเลือกโมเดล และการแคชช่วยให้ควบคุมค่าใช้จ่ายได้โดยไม่ลดทอนคุณภาพ


## สร้างเอเจนต์ที่สังเกตได้

เรานิยามเครื่องมือการเดินทางและห่อการเรียกเอเจนต์ด้วยการจับเวลาเพื่อให้เราสามารถตรวจสอบความหน่วงได้ ในการใช้งานจริงคุณจะรวมเข้ากับ OpenTelemetry หรือแบ็กเอนด์สำหรับการติดตามที่คล้ายกัน


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## รูปแบบการประเมิน

รูปแบบการผลิตทั่วไปคือการใช้เอเจนต์ตัวที่สองเป็น **ผู้ประเมิน** ผู้ประเมินให้คะแนนการตอบของเอเจนต์หลักตามเกณฑ์ที่กำหนดไว้ล่วงหน้า เช่น ความครบถ้วน ความถูกต้อง และความเป็นประโยชน์

สิ่งนี้ช่วยให้:
- เกณฑ์คุณภาพอัตโนมัตก่อนที่การตอบจะถึงผู้ใช้
- การตรวจจับการถดถอยเมื่อพรอมต์หรือโมเดลเปลี่ยนแปลง
- การติดตามประสิทธิภาพของเอเจนต์อย่างต่อเนื่องเมื่อเวลาผ่านไป


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## กลยุทธ์การจัดการต้นทุน

การควบคุมต้นทุนมีความสำคัญสำหรับเอเจนต์ AI ในการผลิต ต่อไปนี้เป็นกลยุทธ์สำคัญ:

| Strategy | Description |
|---|---|
| **Prompt optimization** | ทำให้คำสั่งระบบกระชับ ลบบริบทที่ซ้ำซ้อนเพื่อลดจำนวนโทเค็นป้อนเข้า. |
| **Model selection** | ใช้โมเดลที่มีขนาดเล็กและราคาถูกกว่า (เช่น GPT-4o-mini) สำหรับงานง่าย ๆ เช่น การจำแนกหรือการสกัดข้อมูล และสงวนโมเดลที่ใหญ่กว่าไว้สำหรับการให้เหตุผลที่ซับซ้อน. |
| **Caching** | แคชผลลัพธ์ของเครื่องมือและคำร้องที่พบบ่อยเพื่อหลีกเลี่ยงการเรียก API ซ้ำ ๆ. |
| **Token budgets** | ตั้งขีดจำกัด `max_tokens` เพื่อป้องกันคำตอบที่ยาวโดยไม่คาดคิด. |
| **Batching** | รวบรวมคำถามของผู้ใช้หลายรายการให้เป็นการเรียก API เดียวเมื่อเป็นไปได้. |

ในการปฏิบัติ วิธีการแบบเป็นชั้นมักได้ผลดี: ส่งคำขอที่ตรงไปตรงมาถึงโมเดลที่เร็วและราคาถูก และยกระดับเฉพาะคำถามที่ซับซ้อนไปยังโมเดลที่มีความสามารถมากกว่า.


## Summary

ในบทเรียนนี้ คุณได้เรียนรู้วิธีการ:

1. **เพิ่มความสามารถในการสังเกต (observability)** ให้กับการโต้ตอบของเอเยนต์ด้วยการจับเวลาและการบันทึกล็อก ซึ่งปูพื้นฐานสำหรับการติดตามและการมอนิเตอร์
2. **ประเมินการตอบของเอเยนต์** โดยอัตโนมัติด้วยเอเยนต์ผู้ประเมินที่ให้คะแนนความครบถ้วน ความถูกต้อง และความเป็นประโยชน์
3. **จัดการต้นทุน** ผ่านการปรับแต่งพรอมต์ การเลือกโมเดล การแคช และงบประมาณโทเค็น

รูปแบบการนำไปใช้ในงานจริงเหล่านี้ช่วยให้มั่นใจว่าเอเยนต์ AI ของคุณเชื่อถือได้ วัดผลได้ และคุ้มค่าต่อค่าใช้จ่ายเมื่อใช้งานในระดับขนาดใหญ่


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ข้อจำกัดความรับผิดชอบ:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลด้วยปัญญาประดิษฐ์ Co-op Translator (https://github.com/Azure/co-op-translator) แม้เราจะมุ่งมั่นเพื่อความถูกต้อง โปรดทราบว่าการแปลแบบอัตโนมัติอาจมีข้อผิดพลาดหรือความคลาดเคลื่อน เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เป็นเกณฑ์อ้างอิง สำหรับข้อมูลสำคัญ แนะนำให้ใช้การแปลโดยนักแปลมืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดใดๆ ที่เกิดจากการใช้การแปลฉบับนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
